# 05 Silver DLT Pipeline — Integration Layer

**Laag:** Silver / Integration  
**Schema:** `INTEGRATION`  
**Pipeline type:** Lakeflow Declarative Pipeline (DLT)

## Wat doet deze pipeline?

Leest van Bronze (`STAGING_AZURESTORAGE`) via **Change Data Feed** en mergt de
change-events declaratief via **`apply_changes`** naar Silver Streaming Tables.
Bronze-overschrijvingen (full-load mode) verschijnen als `delete_row` + `insert_row`
events in de CDF en worden clean verwerkt — geen Silver-dubbelen, geen pipeline-break
bij een mode-switch in Bronze.  Zie ADR 0002 en `CONTEXT.md §7`.

De `sales_line` Materialised View joint de twee gecleande Silver-tabellen in batch
(MV-semantiek via `spark.read`) — zie `CONTEXT.md §7` voor de keuze MV boven
stream-stream join.

## Tabellen in deze pipeline (slices #18 + #19 + #20 + #21)

| Tabel | Type | Inhoud |
|---|---|---|
| `INTEGRATION.order_header` | Streaming Table | Gecleaned `order_header` — type-fixes + snake_case + warn-Expectations + drop-rule filter |
| `INTEGRATION.order_header_quarantine` | Streaming Table | Rijen die ten minste één drop-rule violeren, met `failed_rules ARRAY<STRING>` |
| `INTEGRATION.order_detail` | Streaming Table | Gecleaned `order_detail` — type-fixes + snake_case + warn-Expectations + drop-rule filter |
| `INTEGRATION.order_detail_quarantine` | Streaming Table | Rijen die ten minste één drop-rule violeren, met `failed_rules ARRAY<STRING>` |
| `INTEGRATION.sales_line` | Materialised View | Geïntegreerde view: `order_header ⨝ order_detail` op `order_id`, één rij per orderregel |

## Type-fixes (Bronze → Silver)

### order_header

| Bronze-kolom | Bronze-type | Silver-kolom | Silver-type |
|---|---|---|---|
| `SERVED_TS` | `StringType` | `served_ts` | `TimestampType` |
| `ORDER_TAX_AMOUNT` | `StringType` | `order_tax_amount` | `DecimalType(38,4)` |
| `ORDER_DISCOUNT_AMOUNT` | `StringType` | `order_discount_amount` | `DecimalType(38,4)` |
| `LOCATION_ID` | `DoubleType` | `location_id` | `DecimalType(38,0)` |
| `DISCOUNT_ID` | `StringType` | `discount_id` | `DecimalType(38,0)` nullable |
| `SHIFT_START_TIME` | `IntegerType` (millis) | `shift_start_time` | `StringType` `'HH:mm:ss'` |
| `SHIFT_END_TIME` | `IntegerType` (millis) | `shift_end_time` | `StringType` `'HH:mm:ss'` |

### order_detail

| Bronze-kolom | Bronze-type | Silver-kolom | Silver-type |
|---|---|---|---|
| `ORDER_ITEM_DISCOUNT_AMOUNT` | `StringType` | `order_item_discount_amount` | `DecimalType(38,4)` |

All other columns: same type, renamed to snake_case.
All five Bronze audit columns carry through unchanged.

## Severity model (CONTEXT.md §7)

- **fail** — `@expect_all_or_fail`; pipeline halts on violation. Applied to the
  type-fixed Bronze CDF view so it catches violations before clean/quarantine routing.
- **drop** — row is filtered out of the cleansed branch and routed to
  `_quarantine` with a `failed_rules` array.
- **warn** — `@expect_all`; row stays in cleansed, violation surfaces in the DLT
  event log and pipeline metrics.

In [ ]:
%run ./_lib/type_helpers

In [ ]:
%run ./_lib/bronze_cdf

In [ ]:
%run ./_lib/rule_engine

In [ ]:
import dlt
from pyspark.sql import functions as F

## Parameters

DLT reads pipeline parameters from `spark.conf`.  The `catalog` parameter is passed
by the DAB pipeline configuration (`resources/dlt_integration.yml`).

In [ ]:
catalog = spark.conf.get("pipeline.catalog", "DEMO")
bronze_schema = "STAGING_AZURESTORAGE"

print(f"Catalog : {catalog}")
print(f"Bronze  : {catalog}.{bronze_schema}")

## Quality rules — order_header

Three severity levels (CONTEXT.md §7):

- **fail** — `@expect_all_or_fail`; pipeline halts on violation. Attached to the
  type-fixed Bronze CDF view (catches violations regardless of clean/quarantine routing).
- **drop** — row is filtered out of the cleansed branch and routed to
  `order_header_quarantine` with a `failed_rules` array.
- **warn** — `@expect_all`; row stays in cleansed, violation surfaces in DLT
  event metrics. Attached to the cleansed branch view.

Rule names use `<column>_<predicate>` form so `failed_rules` is self-describing
when an analyst queries the quarantine table.

In [ ]:
ORDER_HEADER_FAIL_RULES = {
    "order_id_not_null":         "order_id IS NOT NULL",
}

ORDER_HEADER_DROP_RULES = {
    "order_ts_not_null":         "order_ts IS NOT NULL",
    "customer_id_not_null":      "customer_id IS NOT NULL",
    "order_currency_not_null":   "order_currency IS NOT NULL",
    "order_total_non_negative":  "order_total >= 0",
    "order_amount_non_negative": "order_amount >= 0",
}

ORDER_HEADER_WARN_RULES = {
    "truck_id_not_null":         "truck_id IS NOT NULL",
    "location_id_not_null":      "location_id IS NOT NULL",
    "shift_times_ordered":       "shift_start_time <= shift_end_time",
}

ORDER_HEADER_CLEAN_PREDICATE = build_clean_predicate(ORDER_HEADER_DROP_RULES)

print(f"clean predicate: {ORDER_HEADER_CLEAN_PREDICATE}")

## order_header — cleansed Streaming Table

**Pattern:**
1. `dlt.create_streaming_table` declares the target with `apply_changes` semantics.
2. A `@dlt.view` reads Bronze CDF and applies type-fixes + snake_case rename.
3. `dlt.apply_changes` wires the view into the target Streaming Table.

The CDF `_change_type` column drives the MERGE: `insert_row` / `update_postimage`
become upserts; `delete_row` / `update_preimage` drive deletes.  `truncate_row`
events (from full-load overwrites) are handled as deletes by `apply_changes`.

In [ ]:
dlt.create_streaming_table(
    name="order_header",
    comment="Cleansed order_header: type-fixes (CONTEXT.md §7), snake_case columns, CDF+apply_changes from Bronze.",
    table_properties={
        "delta.enableChangeDataFeed": "true",
    },
)

In [ ]:
@dlt.view(name="order_header_bronze_cdf")
@dlt.expect_all_or_fail(ORDER_HEADER_FAIL_RULES)
def order_header_bronze_cdf():
    """Read Bronze order_header via CDF and apply type-fixes + snake_case rename.

    Returns a streaming DataFrame with:
    - All business columns renamed to snake_case with type corrections
    - All five Bronze audit columns carried through unchanged
    - CDF metadata columns (_change_type, _commit_version, _commit_timestamp)
      retained for apply_changes routing

    The fail-rule expectation halts the pipeline on `order_id IS NULL` regardless
    of which downstream branch (clean / quarantine) the row would otherwise land in.
    """
    src = read_bronze_cdf(f"{catalog}.{bronze_schema}.order_header")

    return (
        src
        # ----------------------------------------------------------------
        # Business columns — type-fixes (CONTEXT.md §7) + snake_case
        # ----------------------------------------------------------------
        .withColumn("order_id",               F.col("ORDER_ID").cast("decimal(38,0)"))
        .withColumn("truck_id",               F.col("TRUCK_ID").cast("decimal(38,0)"))
        .withColumn("location_id",            cast_decimal_38_0(F.col("LOCATION_ID")))   # Double → Decimal
        .withColumn("customer_id",            F.col("CUSTOMER_ID").cast("decimal(38,0)"))
        .withColumn("discount_id",            cast_decimal_38_0(F.col("DISCOUNT_ID")))   # String → Decimal nullable
        .withColumn("shift_id",               F.col("SHIFT_ID").cast("decimal(38,0)"))
        .withColumn("shift_start_time",       millis_to_hhmmss(F.col("SHIFT_START_TIME")))  # Int millis → HH:mm:ss
        .withColumn("shift_end_time",         millis_to_hhmmss(F.col("SHIFT_END_TIME")))    # Int millis → HH:mm:ss
        .withColumn("order_channel",          F.col("ORDER_CHANNEL"))
        .withColumn("order_ts",               F.col("ORDER_TS"))
        .withColumn("served_ts",              cast_served_ts(F.col("SERVED_TS")))            # String → Timestamp
        .withColumn("order_currency",         F.col("ORDER_CURRENCY"))
        .withColumn("order_amount",           F.col("ORDER_AMOUNT").cast("decimal(38,4)"))
        .withColumn("order_tax_amount",       cast_decimal_38_4(F.col("ORDER_TAX_AMOUNT")))        # String → Decimal
        .withColumn("order_discount_amount",  cast_decimal_38_4(F.col("ORDER_DISCOUNT_AMOUNT")))   # String → Decimal
        .withColumn("order_total",            F.col("ORDER_TOTAL").cast("decimal(38,4)"))
        # ----------------------------------------------------------------
        # Bronze audit columns — carried through unchanged (CONTEXT.md §5)
        # ----------------------------------------------------------------
        .withColumn("_ingestion_timestamp",  F.col("_ingestion_timestamp"))
        .withColumn("_source_system",        F.col("_source_system"))
        .withColumn("_source_file",          F.col("_source_file"))
        .withColumn("_last_modified",        F.col("_last_modified"))
        .withColumn("_pipeline_run_id",      F.col("_pipeline_run_id"))
        # ----------------------------------------------------------------
        # Select only the Silver columns (drops all uppercase Bronze cols
        # and keeps CDF metadata for apply_changes)
        # ----------------------------------------------------------------
        .select(
            # CDF routing metadata
            "_change_type",
            "_commit_version",
            "_commit_timestamp",
            # Business columns (snake_case, type-fixed)
            "order_id",
            "truck_id",
            "location_id",
            "customer_id",
            "discount_id",
            "shift_id",
            "shift_start_time",
            "shift_end_time",
            "order_channel",
            "order_ts",
            "served_ts",
            "order_currency",
            "order_amount",
            "order_tax_amount",
            "order_discount_amount",
            "order_total",
            # Audit columns (Bronze passthrough)
            "_ingestion_timestamp",
            "_source_system",
            "_source_file",
            "_last_modified",
            "_pipeline_run_id",
        )
    )

In [ ]:
@dlt.view(name="order_header_clean")
@dlt.expect_all(ORDER_HEADER_WARN_RULES)
def order_header_clean():
    """Cleansed branch — rows that pass every drop rule.

    Filters the type-fixed Bronze CDF view by clean_predicate. Warn-rule violations
    on rows that survive this filter surface in the DLT event log via @expect_all
    (no rows dropped). Fail rules are attached to `order_header_bronze_cdf` so they
    halt the pipeline regardless of branch.
    """
    return dlt.read_stream("order_header_bronze_cdf").filter(ORDER_HEADER_CLEAN_PREDICATE)

In [ ]:
dlt.apply_changes(
    target="order_header",
    source="order_header_clean",
    keys=["order_id"],
    sequence_by="_commit_version",
    apply_as_deletes=F.expr("_change_type = 'delete'"),
    apply_as_truncates=F.expr("_change_type = 'truncate'"),
    except_column_list=["_change_type", "_commit_version", "_commit_timestamp"],
    stored_as_scd_type=1,
)

## order_header_quarantine — Pattern A inverse-filter sink

Same shape as the cleansed branch, but the source view filters on `NOT (clean_predicate)`
and adds a `failed_rules ARRAY<STRING>` column listing every drop rule the row violated.
Both branches feed off the same Bronze CDF view, so the DLT graph shows them as siblings.

Analyst triage example:
```sql
SELECT *
FROM   INTEGRATION.order_header_quarantine
WHERE  array_contains(failed_rules, 'order_total_non_negative');
```

In [ ]:
dlt.create_streaming_table(
    name="order_header_quarantine",
    comment="Quarantine — order_header rows that failed at least one drop rule. failed_rules lists which.",
    table_properties={
        "delta.enableChangeDataFeed": "true",
    },
)

In [ ]:
@dlt.view(name="order_header_quarantine_src")
def order_header_quarantine_src():
    """Quarantine branch — rows that fail at least one drop rule.

    Reads the type-fixed Bronze CDF view, filters to the inverse of clean_predicate,
    and adds `failed_rules ARRAY<STRING>` listing every drop rule a row violated.
    """
    return (
        dlt.read_stream("order_header_bronze_cdf")
        .filter(f"NOT ({ORDER_HEADER_CLEAN_PREDICATE})")
        .withColumn("failed_rules", build_failed_rules_expr(ORDER_HEADER_DROP_RULES))
    )

In [ ]:
dlt.apply_changes(
    target="order_header_quarantine",
    source="order_header_quarantine_src",
    keys=["order_id"],
    sequence_by="_commit_version",
    apply_as_deletes=F.expr("_change_type = 'delete'"),
    apply_as_truncates=F.expr("_change_type = 'truncate'"),
    except_column_list=["_change_type", "_commit_version", "_commit_timestamp"],
    stored_as_scd_type=1,
)

## Quality rules — order_detail

Three severity levels (CONTEXT.md §7), same pattern as order_header:

- **fail** — `@expect_all_or_fail`; pipeline halts on violation. Attached to the
  type-fixed Bronze CDF view.
- **drop** — row is filtered out of the cleansed branch and routed to
  `order_detail_quarantine` with a `failed_rules` array.
- **warn** — `@expect_all`; row stays in cleansed, violation surfaces in DLT
  event metrics.

Rule names use `<column>_<predicate>` form so `failed_rules` is self-describing.

In [ ]:
ORDER_DETAIL_FAIL_RULES = {
    "order_detail_id_not_null": "order_detail_id IS NOT NULL",
}

ORDER_DETAIL_DROP_RULES = {
    "order_id_not_null":        "order_id IS NOT NULL",
    "menu_item_id_not_null":    "menu_item_id IS NOT NULL",
    "quantity_positive":        "quantity > 0",
    "unit_price_non_negative":  "unit_price >= 0",
    "price_non_negative":       "price >= 0",
}

ORDER_DETAIL_WARN_RULES = {
    "line_number_positive":     "line_number > 0",
}

ORDER_DETAIL_CLEAN_PREDICATE = build_clean_predicate(ORDER_DETAIL_DROP_RULES)

print(f"clean predicate: {ORDER_DETAIL_CLEAN_PREDICATE}")

## order_detail — cleansed Streaming Table

**Pattern:** Identical to order_header — Bronze CDF view with type-fixes, clean branch
filtered by drop-rule predicate, quarantine branch with inverse filter + `failed_rules`.

Type fix for order_detail (CONTEXT.md §7):
- `ORDER_ITEM_DISCOUNT_AMOUNT` (StringType) → `order_item_discount_amount` (DecimalType(38,4))

All other columns: same type, renamed to snake_case.

In [ ]:
dlt.create_streaming_table(
    name="order_detail",
    comment="Cleansed order_detail: type-fix ORDER_ITEM_DISCOUNT_AMOUNT→decimal(38,4), snake_case columns, CDF+apply_changes from Bronze.",
    table_properties={
        "delta.enableChangeDataFeed": "true",
    },
)

In [ ]:
@dlt.view(name="order_detail_bronze_cdf")
@dlt.expect_all_or_fail(ORDER_DETAIL_FAIL_RULES)
def order_detail_bronze_cdf():
    """Read Bronze order_detail via CDF and apply type-fixes + snake_case rename.

    Returns a streaming DataFrame with:
    - All business columns renamed to snake_case with type corrections
    - All five Bronze audit columns carried through unchanged
    - CDF metadata columns (_change_type, _commit_version, _commit_timestamp)
      retained for apply_changes routing

    The fail-rule expectation halts the pipeline on `order_detail_id IS NULL`
    regardless of which downstream branch (clean / quarantine) the row would land in.
    """
    src = read_bronze_cdf(f"{catalog}.{bronze_schema}.order_detail")

    return (
        src
        # ----------------------------------------------------------------
        # Business columns — type-fix + snake_case (CONTEXT.md §7)
        # ----------------------------------------------------------------
        .withColumn("order_detail_id",            F.col("ORDER_DETAIL_ID").cast("decimal(38,0)"))
        .withColumn("order_id",                   F.col("ORDER_ID").cast("decimal(38,0)"))
        .withColumn("menu_item_id",               F.col("MENU_ITEM_ID").cast("decimal(38,0)"))
        .withColumn("quantity",                   F.col("QUANTITY").cast("decimal(38,4)"))
        .withColumn("unit_price",                 F.col("UNIT_PRICE").cast("decimal(38,4)"))
        .withColumn("price",                      F.col("PRICE").cast("decimal(38,4)"))
        .withColumn("order_item_discount_amount", cast_decimal_38_4(F.col("ORDER_ITEM_DISCOUNT_AMOUNT")))  # String → Decimal
        .withColumn("line_number",                F.col("LINE_NUMBER").cast("decimal(38,0)"))
        # ----------------------------------------------------------------
        # Bronze audit columns — carried through unchanged (CONTEXT.md §5)
        # ----------------------------------------------------------------
        .withColumn("_ingestion_timestamp",  F.col("_ingestion_timestamp"))
        .withColumn("_source_system",        F.col("_source_system"))
        .withColumn("_source_file",          F.col("_source_file"))
        .withColumn("_last_modified",        F.col("_last_modified"))
        .withColumn("_pipeline_run_id",      F.col("_pipeline_run_id"))
        # ----------------------------------------------------------------
        # Select only the Silver columns (drops all uppercase Bronze cols
        # and keeps CDF metadata for apply_changes)
        # ----------------------------------------------------------------
        .select(
            # CDF routing metadata
            "_change_type",
            "_commit_version",
            "_commit_timestamp",
            # Business columns (snake_case, type-fixed)
            "order_detail_id",
            "order_id",
            "menu_item_id",
            "quantity",
            "unit_price",
            "price",
            "order_item_discount_amount",
            "line_number",
            # Audit columns (Bronze passthrough)
            "_ingestion_timestamp",
            "_source_system",
            "_source_file",
            "_last_modified",
            "_pipeline_run_id",
        )
    )

In [ ]:
@dlt.view(name="order_detail_clean")
@dlt.expect_all(ORDER_DETAIL_WARN_RULES)
def order_detail_clean():
    """Cleansed branch — rows that pass every drop rule.

    Filters the type-fixed Bronze CDF view by clean_predicate. Warn-rule violations
    on rows that survive this filter surface in the DLT event log via @expect_all
    (no rows dropped). Fail rules are attached to `order_detail_bronze_cdf` so they
    halt the pipeline regardless of branch.
    """
    return dlt.read_stream("order_detail_bronze_cdf").filter(ORDER_DETAIL_CLEAN_PREDICATE)

In [ ]:
dlt.apply_changes(
    target="order_detail",
    source="order_detail_clean",
    keys=["order_detail_id"],
    sequence_by="_commit_version",
    apply_as_deletes=F.expr("_change_type = 'delete'"),
    apply_as_truncates=F.expr("_change_type = 'truncate'"),
    except_column_list=["_change_type", "_commit_version", "_commit_timestamp"],
    stored_as_scd_type=1,
)

## order_detail_quarantine — Pattern A inverse-filter sink

Same shape as the cleansed branch, but the source view filters on `NOT (clean_predicate)`
and adds a `failed_rules ARRAY<STRING>` column listing every drop rule the row violated.
Both branches feed off the same `order_detail_bronze_cdf` view.

Analyst triage example:
```sql
SELECT *
FROM   INTEGRATION.order_detail_quarantine
WHERE  array_contains(failed_rules, 'quantity_positive');
```

In [ ]:
dlt.create_streaming_table(
    name="order_detail_quarantine",
    comment="Quarantine — order_detail rows that failed at least one drop rule. failed_rules lists which.",
    table_properties={
        "delta.enableChangeDataFeed": "true",
    },
)

In [ ]:
@dlt.view(name="order_detail_quarantine_src")
def order_detail_quarantine_src():
    """Quarantine branch — rows that fail at least one drop rule.

    Reads the type-fixed Bronze CDF view, filters to the inverse of clean_predicate,
    and adds `failed_rules ARRAY<STRING>` listing every drop rule a row violated.
    """
    return (
        dlt.read_stream("order_detail_bronze_cdf")
        .filter(f"NOT ({ORDER_DETAIL_CLEAN_PREDICATE})")
        .withColumn("failed_rules", build_failed_rules_expr(ORDER_DETAIL_DROP_RULES))
    )

In [ ]:
dlt.apply_changes(
    target="order_detail_quarantine",
    source="order_detail_quarantine_src",
    keys=["order_detail_id"],
    sequence_by="_commit_version",
    apply_as_deletes=F.expr("_change_type = 'delete'"),
    apply_as_truncates=F.expr("_change_type = 'truncate'"),
    except_column_list=["_change_type", "_commit_version", "_commit_timestamp"],
    stored_as_scd_type=1,
)

## sales_line — Materialised View (integrated business view)

`INTEGRATION.sales_line` is the integrated business view: `order_header ⨝ order_detail`
joined on `order_id`, at line grain (one row per order_detail row), with all
order_header attributes denormalised onto each line.

**Pattern:** Defined as `@dlt.table` reading via `spark.read` (batch, MV semantics)
from the cleansed Silver Streaming Tables.  Quarantined rows are excluded by definition
because they are absent from the cleansed inputs.

**Why MV, not stream-stream join?**  (CONTEXT.md §7)
- Joins are simpler in batch — no watermarks, no state management.
- Late arrivals self-resolve on the next pipeline refresh.
- Header updates (e.g. a corrected `order_total`) propagate to existing `sales_line`
  rows automatically on the next refresh — the MV is fully recomputed each run.

**Prefix rule for join ambiguity:**  `order_header` columns that share a name with
`order_detail` (only `order_id`) are resolved via the explicit join key predicate;
`order_detail.order_id` is dropped from the output — the join key is the
`order_header.order_id` copy.

In [ ]:
@dlt.table(
    name="sales_line",
    comment=(
        "Integrated business view: order_header ⨝ order_detail on order_id, "
        "one row per order line with all header attributes denormalised. "
        "Materialised View — batch spark.read, header updates propagate on next refresh."
    ),
)
def sales_line():
    """Materialised View — order_header ⨝ order_detail at line grain.

    Reads the cleansed Silver Streaming Tables via spark.read (batch semantics).
    DLT treats a @dlt.table that uses spark.read as a Materialised View: it is fully
    recomputed on every pipeline run, so late-arriving detail rows and header corrections
    both resolve automatically without any extra CDC handling.

    Grain: one row per order_detail row (order_detail_id is the unique key).
    All order_header columns are denormalised onto each line.
    Quarantined rows are excluded because they are absent from the cleansed inputs.
    """
    silver_schema = "INTEGRATION"

    oh = spark.read.table(f"{catalog}.{silver_schema}.order_header")
    od = spark.read.table(f"{catalog}.{silver_schema}.order_detail")

    return (
        od.join(oh, on="order_id", how="inner")
        .select(
            # -----------------------------------------------------------------
            # Line-grain key
            # -----------------------------------------------------------------
            od.order_detail_id,
            od.order_id,          # join key — kept from order_detail side
            # -----------------------------------------------------------------
            # order_detail business columns
            # -----------------------------------------------------------------
            od.menu_item_id,
            od.quantity,
            od.unit_price,
            od.price,
            od.order_item_discount_amount,
            od.line_number,
            # -----------------------------------------------------------------
            # order_header business columns (denormalised onto each line)
            # -----------------------------------------------------------------
            oh.truck_id,
            oh.location_id,
            oh.customer_id,
            oh.discount_id,
            oh.shift_id,
            oh.shift_start_time,
            oh.shift_end_time,
            oh.order_channel,
            oh.order_ts,
            oh.served_ts,
            oh.order_currency,
            oh.order_amount,
            oh.order_tax_amount,
            oh.order_discount_amount,
            oh.order_total,
        )
    )

## Demo notes

- **DLT graph view**: The Databricks UI shows five nodes:
  `order_header` + `order_header_quarantine` (from `order_header_bronze_cdf`),
  `order_detail` + `order_detail_quarantine` (from `order_detail_bronze_cdf`), and
  `sales_line` reading from both `order_header` and `order_detail` (clean tables only).
- **Mode-switch test**: Switching Bronze `order_detail` between `full` and `incremental`
  and rerunning the workflow will NOT cause Silver row duplication or pipeline failure.
  Full-load overwrites generate `delete_row` + `insert_row` events in CDF, and
  `apply_changes` (SCD Type 1 MERGE) keeps Silver consistent.
- **MV propagation demo**: Update an `order_header` row (e.g. correct an `order_total`).
  After the next pipeline run, the corresponding `sales_line` rows reflect the corrected
  value — the MV is fully recomputed on each refresh.
- **Fail demo**: Insert a Bronze row with `ORDER_DETAIL_ID = NULL` — `expect_all_or_fail`
  halts the pipeline.
- **Drop demo**: Insert a Bronze row with `QUANTITY = 0`. After the run, the row
  appears in `INTEGRATION.order_detail_quarantine` with
  `array_contains(failed_rules, 'quantity_positive') = true`, absent from
  `INTEGRATION.order_detail` and therefore also absent from `INTEGRATION.sales_line`.
- **Warn demo**: Insert a Bronze row with `LINE_NUMBER = 0`. The row stays in the
  cleansed table; the violation count surfaces in the DLT event log.
- **No duplication**: Both header and detail quarantine routing use `build_clean_predicate`
  and `build_failed_rules_expr` from `_lib/rule_engine` — zero copy-paste logic.